# Step 4 — Spectral Extraction

## What this step does

The averaged H5 from Step 3 contains PLcam images — 2D detector images where each fiber appears as a horizontal stripe. Step 4 **extracts a 1D spectrum from each fiber** in every grid bin.

The output is a **coupling map FITS**: a 4D array of shape `(map_n, map_n, Nlambda, Nport)` where:
- `map_n × map_n` = sky position grid
- `Nlambda` = spectral channels (wavelength bins)
- `Nport` = number of fibers (38 for FIRST-PL)

```
PLcam image (412 × 20 px)
────────────────────────────
  ══ fiber 0 ══   ← bright stripe
  ══ fiber 1 ══
      ...              →  extract_to_coupling_map()
  ══ fiber 37 ══
────────────────────────────
                        ↓
         coupling_map[iy, ix, :, fib]  = spectrum of fiber `fib`
                                         when PSF was at grid position (ix, iy)
```

This notebook covers **two extraction methods**:
- **Part A** — Trace box extraction (simpler, works with any dataset)
- **Part B** — FIRST-PL optimal extraction (more accurate, requires a spectrum model)

## Setup

In [ ]:
import os, sys

TUTORIAL_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(TUTORIAL_DIR, '..', '..'))

# ── Input: averaged H5 from Step 3 ───────────────────────────────────────────
MAP_H5_NEW = os.path.join(TUTORIAL_DIR, 'tutorial_output_new', 'step3_map.h5')
MAP_H5_PRE = os.path.join(TUTORIAL_DIR, 'tutorial_output', 'map.h5')
MAP_H5 = MAP_H5_NEW if os.path.exists(MAP_H5_NEW) else MAP_H5_PRE
print(f'Using averaged H5: {MAP_H5}')

# ── Pre-computed trace file ───────────────────────────────────────────────────
# Traces store the y-pixel center of each fiber at each spectral column.
# This file was built once from a bright calibration frame and can be reused
# across observations taken on the same night (traces don't change much).
TRACES_FILE = os.path.join(TUTORIAL_DIR, 'tutorial_output', 'traces.npz')

# ── Spectrum model (for Part B — FIRST-PL optimal extraction) ─────────────────
MODEL_FILE = os.path.join(TUTORIAL_DIR, 'timestamp_matching_output', 'model.npz')

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = os.path.join(TUTORIAL_DIR, 'tutorial_output_new')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Traces file  : {TRACES_FILE}')
print(f'Model file   : {MODEL_FILE}')
print(f'Output dir   : {OUTPUT_DIR}')

## Load a reference PLcam frame

We'll use the bin with the most frames as the reference — it has the best signal-to-noise.

In [ ]:
import h5py, numpy as np
import matplotlib.pyplot as plt

with h5py.File(MAP_H5, 'r') as f:
    avg_plcam = f['avg_PLcam'][:]          # (map_n, map_n, ny, nx)
    nframes   = f['metadata/nframes'][:]   # (map_n, map_n)
    plcam_roi = tuple(f['metadata/plcam_roi'][:])  # (y0, y1, x0, x1)

print(f'avg_PLcam shape : {avg_plcam.shape}   (map_n, map_n, ny, nx)')
print(f'PLcam ROI       : {plcam_roi}  (y0, y1, x0, x1) in detector coords')
print(f'nframes per bin :')
print(nframes)

# Use the bin with the most frames as the reference
best = np.unravel_index(np.argmax(nframes), nframes.shape)
ref_image = avg_plcam[best[0], best[1]]   # shape: (ny, nx)
print(f'\nReference bin   : {best}  (N={nframes[best]} frames)')
print(f'Reference image shape : {ref_image.shape}')

In [ ]:
# Display the reference PLcam image — fiber stripes are clearly visible
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(ref_image, aspect='auto', origin='lower', cmap='gray',
          vmin=np.percentile(ref_image, 5), vmax=np.percentile(ref_image, 99))
ax.set_title('Reference PLcam image (best bin, averaged)')
ax.set_xlabel('Spectral axis (pixels)')
ax.set_ylabel('Fiber axis (pixels)')
plt.tight_layout()
plt.show()

---

# Part A — Trace Box Extraction

## A1. Load or build fiber traces

A **trace file** stores the y-pixel center of each fiber at each spectral column. It is built once from a bright calibration frame and reused across observations.

We first try to load a pre-existing trace file. If not available, we show how to build one from scratch.

In [ ]:
import PLred.specextract as specextract

NFIB = 38   # number of fibers — fixed by the instrument

if os.path.exists(TRACES_FILE):
    # Load pre-computed traces
    d = np.load(TRACES_FILE, allow_pickle=True)
    ylocs  = d['ylocs']    # (nfib,) peak positions
    traces = d['traces']   # (nfib, nx) trace curves
    print(f'Loaded traces from: {TRACES_FILE}')
    print(f'  ylocs shape  : {ylocs.shape}  (y-pixel center of each fiber)')
    print(f'  traces shape : {traces.shape}  (nfib, nx)')
else:
    print(f'Trace file not found. Building from scratch...')

    # Step 1: find fiber peak positions in the cross-dispersion profile
    # If this raises ValueError (wrong number of peaks), adjust `thres` (lower = more sensitive)
    ylocs = specextract.find_peaks(
        image    = ref_image,
        nfib     = NFIB,
        thres    = 0.03,    # detection threshold (0–1); lower → more sensitive
        min_dist = 6,       # minimum pixel spacing between fibers
        plot     = True,
    )

    # Step 2: fit polynomial trace curves column by column
    traces = specextract.find_traces(
        image       = ref_image,
        nfib        = NFIB,
        ini_ys      = ylocs,
        trace_width = 4,
        poly_deg    = 5,
        plot        = True,
    )
    print(f'Traces shape: {traces.shape}')

### Optional: build traces from scratch

If you have new data and no trace file, run the cells below to build one. You only need to do this once per instrument configuration.

**`find_peaks`** locates fiber peaks in the cross-dispersion profile. If you get a `ValueError`, adjust `thres` (lower = more sensitive) or `min_dist` (minimum pixel separation).

**`find_traces`** fits a polynomial to each fiber's y-position column by column.

In [ ]:
# Uncomment to recompute traces from scratch:
#
# ylocs = specextract.find_peaks(
#     image    = ref_image,
#     nfib     = NFIB,
#     thres    = 0.03,    # <-- tune this if find_peaks reports wrong number of peaks
#     min_dist = 6,
#     plot     = True,
# )
# traces = specextract.find_traces(
#     image       = ref_image,
#     nfib        = NFIB,
#     ini_ys      = ylocs,
#     trace_width = 4,
#     poly_deg    = 5,
#     plot        = True,
# )
print('Traces ready. ylocs:', ylocs[:5], '...')

In [ ]:
# Overlay the fitted traces on the reference image
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(ref_image, aspect='auto', origin='lower', cmap='gray',
          vmin=np.percentile(ref_image, 5), vmax=np.percentile(ref_image, 99))
x_arr = np.arange(ref_image.shape[1])
for fib in range(NFIB):
    ax.plot(x_arr, traces[fib], 'r-', lw=0.5, alpha=0.6)
ax.set_title('Fitted fiber traces overlaid on PLcam image')
ax.set_xlabel('Spectral axis (pixels)')
ax.set_ylabel('Fiber axis (pixels)')
plt.tight_layout()
plt.show()

## A2. Build the trace extractor

`make_trace_extractor` returns a callable that, given any PLcam image, sums the pixel values within a box of width `boxsize` pixels around each trace, returning a `(nfib, nx)` spectrum array.

Note that the extractor is a **regular Python function** — you can call it on any PLcam frame.

In [ ]:
trace_extractor = specextract.make_trace_extractor(
    traces_or_model_file = traces,   # the (nfib, nx) trace array
    boxsize              = 3,        # sum ±3 pixels around each trace center
)

# Test it on the reference image
test_spectrum = trace_extractor(ref_image)   # returns (nfib, nwav)
print(f'Extractor output shape: {test_spectrum.shape}  (nfib, nwav)')
print(f'Fiber 0 spectrum: min={test_spectrum[0].min():.1f}  max={test_spectrum[0].max():.1f}')

In [ ]:
# Show extracted spectra for a few fibers
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for i, fib in enumerate([0, 18, 37]):
    axes[i].plot(test_spectrum[fib])
    axes[i].set_ylabel(f'Fiber {fib}')
axes[-1].set_xlabel('Spectral pixel')
fig.suptitle('Extracted spectra (reference bin, trace box method)')
plt.tight_layout()
plt.show()

## A3. Extract all bins → coupling map FITS

`extract_to_coupling_map` loops over all non-empty grid bins, applies the extractor, and writes a FITS file with 5 extensions:
- **[0]** `spectra` — shape `(map_n, map_n, nwav, nfib)`, the coupling map
- **[1]** `nframes` — number of frames per bin
- **[2]** reserved
- **[3]** `variance` — variance of spectra
- **[4]** `normvar` — normalized variance

In [ ]:
OUTPUT_FITS_TRACE = os.path.join(OUTPUT_DIR, 'step4_couplingmap_trace.fits')

specextract.extract_to_coupling_map(
    averaged_h5  = MAP_H5,
    extractor    = trace_extractor,
    output_fits  = OUTPUT_FITS_TRACE,
    verbose      = True,
)

print('\nTrace extraction complete.')

In [ ]:
from astropy.io import fits as afits
import numpy as np

with afits.open(OUTPUT_FITS_TRACE) as hdul:
    print('FITS extensions:')
    hdul.info()
    spectra = hdul[0].data   # shape: (map_n, map_n, nfib, nwav) — FITS axes reversed on read

# FITS files store axes in Fortran order; astropy reverses them on read.
# Written as (nwav, nfib, map_n, map_n), read back as (map_n, map_n, nfib, nwav).
print(f'\nCoupling map shape: {spectra.shape}  (map_n, map_n, nfib, nwav)')

In [ ]:
import matplotlib.pyplot as plt

# Sum over wavelength (axis=3) → (map_n, map_n, nfib)
total_flux = np.nansum(spectra, axis=3)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, fib in enumerate([0, 18, 37]):
    im = axes[i].imshow(total_flux[:, :, fib], origin='lower', cmap='hot')
    plt.colorbar(im, ax=axes[i])
    axes[i].set_title(f'Fiber {fib} — integrated flux')
    axes[i].set_xlabel('Grid x bin')
    axes[i].set_ylabel('Grid y bin')
plt.suptitle('Coupling maps (trace box extraction)')
plt.tight_layout()
plt.show()

---

# Part B — FIRST-PL Optimal Extraction

The **optimal extractor** uses a pre-measured instrument spectrum model to deblend overlapping fiber profiles and minimize noise. It requires a `model.npz` built from lamp/calibration data (see `visPLred/tutorials/pre2_spectrum_model.ipynb`).

**When to use**: when fibers overlap strongly or when you need the best possible signal-to-noise.

## B1. Build the optimal extractor

`make_FIRSTPL_extractor` takes the model file and the PLcam ROI. It handles the coordinate mapping between the model (full detector) and the stored ROI automatically.

In [ ]:
SKIP_PART_B = not os.path.exists(MODEL_FILE)

if SKIP_PART_B:
    print(f'Model file not found: {MODEL_FILE}')
    print('Skipping Part B. To use optimal extraction, provide a model.npz')
    print('built from calibration data (see visPLred/tutorials/pre2_spectrum_model.ipynb).')
else:
    model = specextract.load_spectrum_model(MODEL_FILE)
    print(f'Model: x=[{model.xmin}, {model.xmax}]  ({model.xmax-model.xmin} spectral channels)')
    print(f'       ny_full={model.ny_full}  trace_vals={model.trace_vals.shape}')

    optimal_extractor = specextract.make_FIRSTPL_extractor(
        model_file  = MODEL_FILE,
        plcam_roi   = plcam_roi,   # (y0, y1, x0, x1) stored in the averaged H5
        # dark = None  — dark was already subtracted in Step 2
    )

    # Test on the reference image
    test_optimal = optimal_extractor(ref_image)
    print(f'\nOptimal extractor output: {test_optimal.shape}  (nfib, nwav)')

## B2. Extract all bins and compare

In [ ]:
if not SKIP_PART_B:
    OUTPUT_FITS_OPT = os.path.join(OUTPUT_DIR, 'step4_couplingmap_optimal.fits')

    specextract.extract_to_coupling_map(
        averaged_h5  = MAP_H5,
        extractor    = optimal_extractor,
        output_fits  = OUTPUT_FITS_OPT,
        verbose      = True,
    )
    print('Optimal extraction complete:', OUTPUT_FITS_OPT)

In [ ]:
if not SKIP_PART_B:
    # Compare spectra from the best bin, fiber 0
    spec_trace   = trace_extractor(ref_image)    # (nfib, nwav_trace)
    spec_optimal = optimal_extractor(ref_image)  # (nfib, nwav_optimal)

    fig, axes = plt.subplots(2, 1, figsize=(10, 6))
    axes[0].plot(spec_trace[0] / spec_trace[0].max(), label='Trace box')
    axes[0].set_ylabel('Normalized flux')
    axes[0].set_title('Fiber 0 spectrum — best bin')
    axes[0].legend()

    axes[1].plot(spec_optimal[0] / spec_optimal[0].max(), label='Optimal (FIRST-PL)', color='orange')
    axes[1].set_xlabel('Spectral channel')
    axes[1].set_ylabel('Normalized flux')
    axes[1].legend()

    plt.tight_layout()
    plt.show()

## Summary

Step 4 produced the **coupling map FITS** — the core output of the data reduction pipeline.

| Method | Output file | When to use |
|--------|------------|-------------|
| Trace box | `step4_couplingmap_trace.fits` | Quick, no model needed, works for all data |
| Optimal (FIRST-PL) | `step4_couplingmap_optimal.fits` | Best SNR, requires a calibrated spectrum model |

The coupling map FITS is the input to **image reconstruction** — see `step3_image_reconstruction.ipynb`.

---

## CLI equivalent

```ini
[Instrument]
nfib = 38
spectral_orientation = horizontal

[Specextract]
input      = tutorial_output_new/step3_map.h5
extractor  = trace_box
output     = tutorial_output_new/step4_couplingmap_trace.fits
plcam_roi  = "0,412,1200,1220"
trace_file = tutorial_output/traces.npz
truncate   = 0
```

```python
import PLred.specextract as specextract
specextract.extract_from_config('obs.ini')
# or:
import PLred.pipeline as pipeline
pipeline.run_mode1('obs.ini', steps=[4])
```

For optimal extraction, change `extractor = FIRSTPL_optimal` and add:
```ini
model_file = path/to/model.npz
```

In [ ]:
print('Done.')
print('Trace coupling map :', OUTPUT_FITS_TRACE)
if not SKIP_PART_B:
    print('Optimal coupling map:', OUTPUT_FITS_OPT)